# PEFT Basics: Prompt Tuning Qwen3.5-2B-Base


Практический notebook по Parameter-Efficient Fine-Tuning (PEFT) на `Qwen/Qwen3.5-2B-Base`.

Используется **Prompt Tuning**: веса базовой модели остаются замороженными, а обучаются только виртуальные prompt embeddings. Это позволяет сначала понять принцип PEFT, а затем перейти к LoRA и QLoRA.

Задача - бинарная классификация тональности SST-2, сформулированная как causal language modeling: модель должна генерировать `positive` или `negative`.


## Теоретическая часть


### Что такое fine-tuning


Предобученная языковая модель уже умеет работать с языком и хранит большое количество общих знаний, но это не означает, что она оптимально решает конкретную прикладную задачу.

**Fine-tuning** — это продолжение обучения pretrained model на специализированных данных. Он позволяет адаптировать модель к определённому домену, формату ответа, стилю, классификации, извлечению информации или другой downstream-задаче без обучения модели с нуля.


### Full fine-tuning


При классическом **full fine-tuning** практически все параметры модели остаются trainable (изменяются).

Градиенты проходят через всю сеть, optimizer обновляет все или почти все веса, а для каждой новой задачи обычно получается отдельная полностью дообученная копия модели.

Упрощённо:

```text
Pretrained model
      │
      ▼
Все параметры trainable
      │
      ▼
Training
      │
      ▼
Полная новая версия модели
```


### Почему full fine-tuning дорогой


Для моделей с миллиардами параметров стоимость обучения определяется не только размером файла с весами.

Во время обучения в GPU memory находятся:

- параметры модели;
- gradients;
- optimizer states;
- activations;
- временные tensors и buffers.

Например, 2 миллиарда параметров в BF16 занимают примерно 4 GB только для хранения самих весов. Реальный training footprint значительно больше.

Кроме того, если одну модель адаптировать к десяти задачам через full fine-tuning, приходится хранить десять почти полных копий модели.


### Что такое PEFT


**PEFT — Parameter-Efficient Fine-Tuning** — семейство методов, которые позволяют адаптировать большие pretrained models, обучая только небольшую часть параметров.

Главная идея:

```text
Большая pretrained model
        │
        ├── большая часть параметров frozen
        │
        └── небольшая часть параметров trainable
                         │
                         ▼
                      Training
                         │
                         ▼
                   Небольшой adapter
```

Одна базовая модель может использоваться совместно с множеством небольших task-specific adapters.


### Что именно экономит PEFT


PEFT уменьшает:

- число trainable parameters;
- память для gradients;
- память для optimizer states;
- размер task-specific checkpoints;
- стоимость хранения нескольких специализированных версий модели.

Но важно понимать ограничение: frozen base model всё равно участвует в forward pass и обычно должна находиться в GPU memory. PEFT не превращает большую модель в маленькую.


### PEFT как идея и библиотека `peft`


PEFT — это общая концепция, существовавшая до появления библиотеки Hugging Face.

Например, adapter-based parameter-efficient transfer learning для Transformers был опубликован ещё в 2019 году.

Библиотека **`peft`** от Hugging Face предоставляет единый API для большого количества таких методов и интегрируется с `transformers`.

Типичный workflow:

```text
AutoModel...
    │
    ▼
PEFT configuration
    │
    ▼
get_peft_model(...)
    │
    ▼
PeftModel
    │
    ▼
Trainer
    │
    ▼
adapter checkpoint
```


### Основные семейства PEFT


| Семейство | Что обучается | Примеры |
|---|---|---|
| Soft prompting | Виртуальные trainable embeddings | Prompt Tuning, Prefix Tuning, P-Tuning |
| Low-rank adaptation | Небольшие low-rank matrices | LoRA, AdaLoRA |
| Adapter methods | Компактные дополнительные transformations | IA3 и другие adapters |
| Selective tuning | Только выбранные существующие параметры | Trainable tokens, LayerNorm tuning |

LoRA — только один из PEFT-методов, хотя сегодня он является наиболее распространённым.


### Краткая хронология


| Год | Метод | Основная идея |
|---:|---|---|
| 2019 | Adapters | Небольшие trainable modules внутри frozen Transformer |
| 2021 | Prefix Tuning | Trainable continuous prefixes |
| 2021 | Prompt Tuning | Trainable soft prompt embeddings |
| 2021 | P-Tuning | Continuous prompts с prompt encoder |
| 2021 | LoRA | Low-rank decomposition обновления весов |
| 2023 | QLoRA | LoRA поверх 4-bit quantized base model |

В этом notebook используется **Prompt Tuning**, потому что на нём особенно наглядно видно базовый принцип PEFT.


### Hard prompt и soft prompt


Обычная текстовая инструкция — это **hard prompt**:

```text
Classify the sentiment as positive or negative.
```

Она состоит из реальных токенов словаря и выбирается человеком.

**Soft prompt** состоит из trainable vectors в embedding space. Эти vectors не обязаны соответствовать каким-либо словам и изменяются через backpropagation.


### Как работает Prompt Tuning


Пусть обычный tokenized input после embedding layer выглядит так:

```text
x1  x2  x3  ...  xn
```

Prompt Tuning добавляет перед ним обучаемые virtual tokens:

```text
p1  p2  ...  pk  x1  x2  x3  ...  xn
```

где:

```text
p1 ... pk  → trainable
x1 ... xn  → обычные input embeddings
base model → frozen
```

Во время training optimizer изменяет только `p1 ... pk`.


### Full fine-tuning и Prompt Tuning визуально


```text
FULL FINE-TUNING

Task A ──► [ entire model A ]
Task B ──► [ entire model B ]
Task C ──► [ entire model C ]

Для каждой задачи хранится полная модель.


PROMPT TUNING

Prompt A ─┐
Prompt B ─┼──► [ one frozen base model ]
Prompt C ─┘

Для каждой задачи хранится только маленький soft prompt.
```


### Официальная схема Hugging Face


![Model Tuning vs Prompt Tuning](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/peft/prompt-tuning.png)


Схема показывает главное отличие: при обычном model tuning для каждой задачи появляется собственная дообученная модель, тогда как Prompt Tuning позволяет переиспользовать одну frozen base model и хранить маленькие task-specific prompts.


### Сколько параметров обучается


Для простого Prompt Tuning число trainable parameters примерно равно:

```text
num_virtual_tokens × hidden_size
```

В notebook используется:

```text
num_virtual_tokens = 16
hidden_size        = 2048
```

Поэтому ожидаемый порядок величины:

```text
16 × 2048 = 32 768 trainable parameters
```

на фоне примерно **2 миллиардов параметров** Qwen3.5-2B-Base.

Фактическое число notebook затем вычисляет автоматически через `requires_grad`.


### Почему Prompt Tuning масштабируется


Размер soft prompt зависит в основном от:

- количества virtual tokens;
- hidden size модели.

Он почти не зависит от общего количества Transformer layers.

Поэтому при переходе от 2B-модели к более крупной модели количество обучаемых параметров растёт намного медленнее, чем размер base model.


### Почему взят Qwen3.5-2B-Base


`Qwen/Qwen3.5-2B-Base` подходит для учебного PEFT notebook по нескольким причинам:

- это современная base model;
- размер уже достаточно велик, чтобы PEFT имел практический смысл;
- модель всё ещё достаточно компактна для локальных экспериментов;
- text-only backbone доступен через `AutoModelForCausalLM`;
- тот же workflow можно масштабировать на более крупный checkpoint;
- Prompt Tuning почти не зависит от внутренних имён attention modules.

Последний пункт важен: будущий LoRA notebook будет уже сильнее зависеть от внутренней архитектуры модели и выбора `target_modules`.


### Prompt Tuning, LoRA и QLoRA


| Метод | Base model | Что обучается | Quantization |
|---|---|---|---|
| Full fine-tuning | trainable | почти все параметры | не обязательна |
| Prompt Tuning | frozen | virtual prompt embeddings | не обязательна |
| LoRA | frozen | low-rank matrices | не обязательна |
| QLoRA | frozen + quantized | LoRA matrices | обычно 4-bit |

Учебная последовательность:

```text
PEFT / Prompt Tuning
        ↓
       LoRA
        ↓
bitsandbytes quantization
        ↓
      QLoRA
```


### Что проверим на практике


В практической части теория проверяется измерениями:

1. сколько параметров содержит Qwen3.5-2B-Base;
2. сколько параметров остаётся trainable после Prompt Tuning;
3. действительно ли base model frozen;
4. как отличаются generation-based и forced-choice оценки до и после обучения;
5. accuracy, precision, recall, F1, Macro F1 и confusion matrix;
6. Label Token Accuracy и Perplexity во время validation;
7. как ведут себя validation loss и Linear LR scheduler;
8. размер Prompt Tuning adapter;
9. воспроизводимость сохранённого adapter после clean reload.


### Ограничения Prompt Tuning


Prompt Tuning обучает очень мало параметров, но это не означает, что он всегда лучший вариант.

- Soft prompts не являются человекочитаемыми.
- Качество может зависеть от initialization.
- Важен выбор числа virtual tokens.
- Base model всё равно требует вычислений.
- Activations всё равно занимают память.
- Для многих современных LLM-задач LoRA обычно показывает более сильный и предсказуемый результат.

Поэтому Prompt Tuning здесь используется прежде всего как максимально наглядное введение в PEFT.


### Источники по теории


- Hugging Face PEFT — Soft prompts: https://huggingface.co/docs/peft/main/en/conceptual_guides/prompting
- Hugging Face PEFT — Methods overview: https://huggingface.co/docs/peft/main/methods/overview
- Parameter-Efficient Transfer Learning for NLP: https://arxiv.org/abs/1902.00751
- The Power of Scale for Parameter-Efficient Prompt Tuning: https://arxiv.org/abs/2104.08691
- LoRA: https://arxiv.org/abs/2106.09685
- QLoRA: https://arxiv.org/abs/2305.14314
- Qwen3.5-2B-Base: https://huggingface.co/Qwen/Qwen3.5-2B-Base


## Практическая часть


### 1. Импорты и проверка среды


Зависимости устанавливаются в Docker image, а не внутри notebook.

Notebook только проверяет версии основных библиотек. Для стандартных classification metrics используется `TorchMetrics`, чтобы метрики оставались внутри PyTorch-стека.

In [2]:
import gc
import sys
from contextlib import contextmanager
from dataclasses import dataclass
from importlib.metadata import version as package_version
from pathlib import Path
from typing import Any

import numpy as np
import plotly.graph_objects as go
import torch
from datasets import DatasetDict, load_dataset
from huggingface_hub import HfApi, create_repo
from packaging.version import Version
from torchmetrics import MetricCollection
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassConfusionMatrix,
    MulticlassF1Score,
    MulticlassPrecision,
    MulticlassRecall,
)
from peft import (
    PeftConfig,
    PeftModel,
    PromptTuningConfig,
    PromptTuningInit,
    TaskType,
    get_peft_model,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

print(f"Python:           {sys.version.split()[0]}")
print(f"PyTorch:          {torch.__version__}")
print(f"Transformers:     {package_version('transformers')}")
print(f"PEFT:             {package_version('peft')}")
print(f"Datasets:         {package_version('datasets')}")
print(f"Accelerate:       {package_version('accelerate')}")
print(f"Hugging Face Hub: {package_version('huggingface_hub')}")
print(f"Plotly:           {package_version('plotly')}")
print(f"TorchMetrics:     {package_version('torchmetrics')}")

print(f"\nCUDA available:   {torch.cuda.is_available()}")


Python:           3.10.12
PyTorch:          2.13.0+cu130
Transformers:     5.14.1
PEFT:             0.20.0
Datasets:         5.0.1
Accelerate:       1.14.0
Hugging Face Hub: 1.28.0
Plotly:           6.9.0
TorchMetrics:     1.9.0

CUDA available:   True


In [3]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


In [4]:
def clear_device_memory():
    """Release Python garbage and unused CUDA allocator cache."""
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()


### 2. Конфигурация


Notebook поддерживает два режима:

```text
RUN_MODE = "smoke"  → 4 000 training examples
RUN_MODE = "full"   → весь SST-2 train split
```

`smoke` используется по умолчанию и сохраняет ранее проверенный масштаб Prompt Tuning experiment.

При effective batch size 16 одна эпоха на 4 000 примерах содержит примерно

$\frac{4000}{16} = 250$

optimizer steps, а три эпохи дают примерно

$250 \times 3 = 750$

optimizer steps.

Validation split остаётся полным.

Linear LR scheduler и более частая validation особенно полезны здесь из-за сравнительно высокого learning rate Prompt Tuning.

Локально сохраняются checkpoints, reproducibility metadata и финальный Prompt Tuning adapter. Training dynamics отображается только внутри notebook.


In [5]:
SEED = 42

MODEL_ID = "Qwen/Qwen3.5-2B-Base"
DATASET_ID = "stanfordnlp/sst2"

MODEL_REVISION = None
DATASET_REVISION = None

TEXT_COLUMN = "sentence"
LABEL_COLUMN = "label"
LABEL_NAMES = {
    0: "negative",
    1: "positive",
}

RUN_MODE = "smoke"
assert RUN_MODE in {"smoke", "full"}

PROMPT_INIT_TEXT = (
    "Classify the sentiment of the movie review "
    "as positive or negative."
)
NUM_VIRTUAL_TOKENS = 16

MAX_LENGTH = 128
MAX_TRAIN_SAMPLES = (
    4_000
    if RUN_MODE == "smoke"
    else None
)
MAX_EVAL_SAMPLES = None

BASELINE_EVAL_SAMPLES = None
FINAL_EVAL_SAMPLES = None
GENERATION_BATCH_SIZE = 32
FORCED_CHOICE_BATCH_SIZE = 16
MAX_NEW_TOKENS = 4

ARTIFACT_RELOAD_SAMPLES = 8
RUN_ARTIFACT_RELOAD_TEST = True

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 1
NUM_TRAIN_EPOCHS = 3

LEARNING_RATE = 3e-2
LR_SCHEDULER_TYPE = "linear"
WARMUP_STEPS = 0.05
WEIGHT_DECAY = 0.0

EVAL_STEPS = 100
SAVE_STEPS = EVAL_STEPS

EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.0

PUSH_TO_HUB = False
HUB_MODEL_ID = (
    "artyomboyko/qwen3.5-2b-sst2-prompt-tuning"
)

cwd = Path.cwd().resolve()

if cwd.name == "peft":
    NOTEBOOK_DIR = cwd
elif (cwd / "notebooks" / "finetuning" / "peft").is_dir():
    NOTEBOOK_DIR = (
        cwd / "notebooks" / "finetuning" / "peft"
    ).resolve()
elif Path("/workspace/notebooks/finetuning/peft").is_dir():
    NOTEBOOK_DIR = Path(
        "/workspace/notebooks/finetuning/peft"
    )
else:
    NOTEBOOK_DIR = cwd

OUTPUT_DIR = (
    NOTEBOOK_DIR
    / "outputs"
    / "qwen3.5-2b-sst2-prompt-tuning"
)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

set_seed(SEED)

if device.type == "cuda":
    torch.set_float32_matmul_precision("high")

print(f"Run mode:             {RUN_MODE}")
print(f"Notebook directory:   {NOTEBOOK_DIR}")
print(f"Output directory:     {OUTPUT_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

RESULTS_TABLE = []


Run mode:             smoke
Notebook directory:   /workspace/notebooks/finetuning/peft
Output directory:     /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-prompt-tuning
Checkpoint directory: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-prompt-tuning/checkpoints


### 3. Загрузка SST-2


Используется Stanford SST-2 с двумя классами: `negative` и `positive`.

Validation split сохраняется полностью. Notebook проверяет это assertion, чтобы случайно не получить оценку только на подмножестве.


In [6]:
raw_dataset = load_dataset(
    DATASET_ID,
    revision=DATASET_REVISION,
)

dataset = DatasetDict(
    train=raw_dataset["train"],
    validation=raw_dataset["validation"],
)


def limit_split(
    split,
    max_samples,
):
    if max_samples is None:
        return split

    count = min(
        max_samples,
        len(split),
    )

    return (
        split
        .shuffle(seed=SEED)
        .select(range(count))
    )


dataset["train"] = limit_split(
    dataset["train"],
    MAX_TRAIN_SAMPLES,
)

dataset["validation"] = limit_split(
    dataset["validation"],
    MAX_EVAL_SAMPLES,
)

print(dataset)
print(dataset["train"][0])

print(
    f"\nTrain samples:      "
    f"{len(dataset['train']):,}"
)

print(
    f"Validation samples: "
    f"{len(dataset['validation']):,} "
    f"/ {len(raw_dataset['validation']):,}"
)

if MAX_EVAL_SAMPLES is None:
    assert (
        len(dataset["validation"])
        == len(raw_dataset["validation"])
    ), (
        "Validation split was unexpectedly "
        "truncated."
    )

dataset_train_fingerprint = (
    dataset["train"]._fingerprint
)
dataset_validation_fingerprint = (
    dataset["validation"]._fingerprint
)

try:
    resolved_dataset_revision = (
        DATASET_REVISION
        or HfApi().dataset_info(
            DATASET_ID
        ).sha
    )
except Exception:
    resolved_dataset_revision = (
        DATASET_REVISION
        or "unavailable"
    )

print(
    f"Dataset revision:   "
    f"{resolved_dataset_revision}"
)
print(
    f"Train fingerprint:  "
    f"{dataset_train_fingerprint}"
)
print(
    f"Valid fingerprint:  "
    f"{dataset_validation_fingerprint}"
)


DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
})
{'idx': 32326, 'sentence': 'klein , charming in comedies like american pie and dead-on in election , ', 'label': 1}

Train samples:      4,000
Validation samples: 872 / 872
Dataset revision:   8d51e7e4887a4caaa95b3fbebbf53c0490b58bbb
Train fingerprint:  cc068cadba0337ea
Valid fingerprint:  c1ddc6497ec97f98


### 4. Tokenizer и Qwen3.5-2B-Base


Модель загружается без quantization.

Prompt Tuning обучает только virtual prompt embeddings, а исходные weights Qwen3.5 остаются базой для адаптера.


In [7]:
use_bf16 = (
    device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

model_dtype = (
    torch.bfloat16
    if use_bf16
    else torch.float16
    if device.type == "cuda"
    else torch.float32
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=model_dtype,
)

base_model = base_model.to(device)

print(
    f"Loaded class: "
    f"{type(base_model).__name__}"
)
print(
    f"Model dtype:  "
    f"{next(base_model.parameters()).dtype}"
)
print(
    f"Vocabulary:   "
    f"{len(tokenizer):,}"
)
print(
    f"EOS token:    "
    f"{tokenizer.eos_token!r}"
)
print(
    f"PAD token:    "
    f"{tokenizer.pad_token!r}"
)

resolved_model_revision = (
    MODEL_REVISION
    or getattr(
        base_model.config,
        "_commit_hash",
        None,
    )
)

if not resolved_model_revision:
    try:
        resolved_model_revision = (
            HfApi()
            .model_info(MODEL_ID)
            .sha
        )
    except Exception:
        resolved_model_revision = (
            "unavailable"
        )

print(
    f"Model revision: "
    f"{resolved_model_revision}"
)


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded class: Qwen3_5ForCausalLM
Model dtype:  torch.bfloat16
Vocabulary:   248,077
EOS token:    '<|endoftext|>'
PAD token:    '<|endoftext|>'
Model revision: b1485b2fa6dfa1287294f269f5fb618e03d52d7c


### 5. Параметры base model


До PEFT считаются параметры исходной модели. После создания Prompt Tuning adapter те же функции покажут, какая доля параметров действительно обучается.


In [8]:
def parameter_stats(model):
    total = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    return total, trainable


def parameter_memory_gib(model):
    total_bytes = sum(
        parameter.numel()
        * parameter.element_size()
        for parameter in model.parameters()
    )

    return total_bytes / (1024 ** 3)


base_total_params, base_trainable_params = (
    parameter_stats(base_model)
)

print(
    f"Total parameters:     "
    f"{base_total_params:,}"
)
print(
    f"Trainable parameters: "
    f"{base_trainable_params:,}"
)
print(
    f"Trainable share:      "
    f"{100 * base_trainable_params / base_total_params:.6f}%"
)


Total parameters:     1,881,825,088
Trainable parameters: 1,881,825,088
Trainable share:      100.000000%


### 6. Формат задачи


SST-2 classification формулируется как causal language modeling.

Loss считается только по токенам целевой метки. Инструкция и review маскируются значением `-100`.


In [9]:
VISIBLE_INSTRUCTION = (
    "Classify the sentiment of this movie review "
    "as positive or negative."
)


def build_prompt(text):
    return (
        f"{VISIBLE_INSTRUCTION}\n"
        f"Review: {text.strip()}\n"
        "Sentiment:"
    )


def preprocess_batch(examples):
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for text, label_id in zip(
        examples[TEXT_COLUMN],
        examples[LABEL_COLUMN],
    ):
        target_text = (
            " "
            + LABEL_NAMES[int(label_id)]
        )

        target_ids = tokenizer(
            target_text,
            add_special_tokens=False,
        )["input_ids"]

        target_ids = (
            target_ids
            + [tokenizer.eos_token_id]
        )

        max_prompt_length = max(
            1,
            MAX_LENGTH - len(target_ids),
        )

        prompt_ids = tokenizer(
            build_prompt(text),
            add_special_tokens=False,
            truncation=True,
            max_length=max_prompt_length,
        )["input_ids"]

        input_ids = (
            prompt_ids
            + target_ids
        )

        labels = (
            [-100] * len(prompt_ids)
            + target_ids
        )

        all_input_ids.append(
            input_ids
        )
        all_attention_masks.append(
            [1] * len(input_ids)
        )
        all_labels.append(
            labels
        )

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels,
    }


processed_dataset = dataset.map(
    preprocess_batch,
    batched=True,
    batch_size=1_000,
    remove_columns=(
        dataset["train"].column_names
    ),
    desc="Tokenizing SST-2",
)

print(processed_dataset)


Tokenizing SST-2:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing SST-2:   0%|          | 0/872 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 872
    })
})


### 7. Data collator


Padding выполняется динамически по текущему batch. Padding в labels заменяется на `-100`, поэтому эти позиции не участвуют в loss.


In [10]:
@dataclass
class CausalClassificationCollator:
    tokenizer: Any

    def __call__(
        self,
        features,
    ):
        model_features = [
            {
                "input_ids": (
                    feature["input_ids"]
                ),
                "attention_mask": (
                    feature["attention_mask"]
                ),
            }
            for feature in features
        ]

        batch = self.tokenizer.pad(
            model_features,
            padding=True,
            return_tensors="pt",
        )

        sequence_length = (
            batch["input_ids"].shape[1]
        )

        padded_labels = []

        for feature in features:
            labels = feature["labels"]

            padding_length = (
                sequence_length
                - len(labels)
            )

            padded_labels.append(
                labels
                + [-100] * padding_length
            )

        batch["labels"] = torch.tensor(
            padded_labels,
            dtype=torch.long,
        )

        return batch


data_collator = (
    CausalClassificationCollator(
        tokenizer=tokenizer,
    )
)

test_batch = data_collator(
    [
        processed_dataset["train"][0],
        processed_dataset["train"][1],
    ]
)

for key, value in test_batch.items():
    print(
        f"{key:16s}: "
        f"{tuple(value.shape)} "
        f"{value.dtype}"
    )


input_ids       : (2, 37) torch.int64
attention_mask  : (2, 37) torch.int64
labels          : (2, 37) torch.int64


### 8. Метрики и функции оценки

Одной accuracy недостаточно, потому что causal LM может понимать правильный label, но выдавать его в неподходящем формате.

Поэтому notebook использует два взаимодополняющих режима оценки: свободную генерацию и forced-choice по вероятности допустимых labels.

#### Generation-based accuracy

**Generation-based accuracy** — доля validation examples, для которых свободно сгенерированная и распознанная метка совпала с reference.

$\mathrm{Accuracy} = \frac{N_{\mathrm{correct}}}{N_{\mathrm{all}}}$

`invalid` output считается неправильным ответом, потому что модель не выполнила output contract.

#### Valid output rate

**Valid output rate** показывает, какая доля генераций распознаётся как одна из допустимых меток `negative` или `positive`.

$\mathrm{ValidOutputRate} = \frac{N_{\mathrm{valid}}}{N_{\mathrm{all}}}$

#### Forced-choice accuracy

Для текущей пары labels tokenizer должен кодировать `" negative"` и `" positive"` одним token. Это позволяет оценивать forced-choice напрямую по next-token logits после prompt, без построения двух полных candidate sequences и ручного суммирования token log-probabilities.

$\hat y = \arg\max_{y \in \{\text{negative},\text{positive}\}} P(y \mid x)$

Такой вариант особенно удобен для Prompt Tuning: virtual tokens добавляются перед prompt, но последний logit по-прежнему описывает распределение следующего token после исходного текста.

#### Precision, Recall, F1 и Macro F1

Стандартные classification metrics объединены в один `TorchMetrics.MetricCollection`. Predictions и references хранятся как integer class IDs, поэтому между evaluation loop и метриками больше нет промежуточного преобразования в строки и обратно. `invalid` используется как отдельный prediction class, а Macro F1 усредняется только по целевым классам `negative` и `positive`.

#### Label Token Accuracy

**Label Token Accuracy** используется как лёгкая training-time метрика. Она рассчитывается только на позициях supervised target; prompt и padding с label `-100` не учитываются. EOS исключается отдельно.

$\mathrm{LabelTokenAccuracy} = \frac{N_{\mathrm{correct\ target\ tokens}}}{N_{\mathrm{target\ tokens}}}$

#### Perplexity

**Perplexity** рассчитывается из validation loss:

$\mathrm{PPL} = e^{L_{\mathrm{eval}}}$

Она не заменяет classification metrics, а показывает среднюю уверенность causal LM в правильных target tokens.

#### Почему используются оба режима

Два режима отвечают на разные вопросы:

```text
Forced-choice
→ понимает ли модель, какой label вероятнее?

Generation-based
→ может ли модель реально выдать этот label в требуемом формате?
```

Их совместная интерпретация отделяет качество классификации от соблюдения output format.

In [11]:
CLASS_LABELS = ("negative", "positive")
INVALID_LABEL_ID = len(CLASS_LABELS)
NUM_EVAL_CLASSES = INVALID_LABEL_ID + 1

EVAL_METRICS = MetricCollection(
    {
        "accuracy": MulticlassAccuracy(NUM_EVAL_CLASSES, average="micro"),
        "precision": MulticlassPrecision(NUM_EVAL_CLASSES, average=None, zero_division=0),
        "recall": MulticlassRecall(NUM_EVAL_CLASSES, average=None, zero_division=0),
        "f1": MulticlassF1Score(NUM_EVAL_CLASSES, average=None, zero_division=0),
        "confusion_matrix": MulticlassConfusionMatrix(NUM_EVAL_CLASSES),
    }
)

_label_token_ids = [
    tokenizer(f" {label}", add_special_tokens=False)["input_ids"]
    for label in CLASS_LABELS
]
if not all(len(ids) == 1 for ids in _label_token_ids):
    raise ValueError("Forced-choice evaluation requires single-token labels.")
LABEL_TOKEN_IDS = torch.tensor([ids[0] for ids in _label_token_ids])


@contextmanager
def temporary_padding_side(tokenizer, padding_side):
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = padding_side
    try:
        yield
    finally:
        tokenizer.padding_side = original_padding_side


def select_eval_split(raw_split, max_samples):
    count = len(raw_split) if max_samples is None else min(max_samples, len(raw_split))
    return raw_split.select(range(count))


def tokenize_eval_batch(texts, device):
    return tokenizer(
        [build_prompt(text) for text in texts],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)


def normalize_prediction(text):
    text = text.strip().lower()
    return next(
        (i for i, label in enumerate(CLASS_LABELS) if text.startswith(label)),
        INVALID_LABEL_ID,
    )


def classification_metrics(predictions, references):
    predictions = torch.as_tensor(predictions, dtype=torch.long)
    references = torch.as_tensor(references, dtype=torch.long)
    metrics = EVAL_METRICS.clone()(predictions, references)
    metrics["macro_f1"] = metrics["f1"][: len(CLASS_LABELS)].mean()
    metrics["total"] = references.numel()
    return metrics


def print_classification_summary(title, metrics):
    print(f"\n{title}\n{'-' * len(title)}")
    print(f"Accuracy:  {metrics['accuracy']:.2%}")
    print(f"Macro F1:  {metrics['macro_f1']:.4f}")
    for i, label in enumerate(CLASS_LABELS):
        print(
            f"{label:8s} precision={metrics['precision'][i]:.4f} "
            f"recall={metrics['recall'][i]:.4f} f1={metrics['f1'][i]:.4f}"
        )
    if "valid_output_rate" in metrics:
        print(f"Valid output rate: {metrics['valid_output_rate']:.2%}")

    print("\nConfusion matrix (rows=actual, columns=predicted)")
    print(f"{'':12s}{'negative':>10s}{'positive':>10s}{'invalid':>10s}")
    for label, row in zip(CLASS_LABELS, metrics["confusion_matrix"][:2]):
        print(f"{label:12s}{row[0]:10d}{row[1]:10d}{row[2]:10d}")


def evaluate_generation(model, raw_split, max_samples=None, batch_size=32):
    model.eval()
    eval_split = select_eval_split(raw_split, max_samples)
    device = next(model.parameters()).device
    predictions, references, examples = [], [], []

    with temporary_padding_side(tokenizer, "left"), torch.inference_mode():
        for start in range(0, len(eval_split), batch_size):
            batch = eval_split[start : start + batch_size]
            inputs = tokenize_eval_batch(batch[TEXT_COLUMN], device)
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            texts = tokenizer.batch_decode(
                outputs[:, inputs["input_ids"].shape[1] :],
                skip_special_tokens=True,
            )
            batch_predictions = [normalize_prediction(text) for text in texts]
            batch_references = [int(label) for label in batch[LABEL_COLUMN]]

            predictions.extend(batch_predictions)
            references.extend(batch_references)
            examples.extend(
                (
                    text,
                    None if pred == INVALID_LABEL_ID else CLASS_LABELS[pred],
                    CLASS_LABELS[ref],
                )
                for text, pred, ref in zip(texts, batch_predictions, batch_references)
            )

    metrics = classification_metrics(predictions, references)
    metrics.update(
        valid_output_rate=sum(p != INVALID_LABEL_ID for p in predictions) / len(eval_split),
        predictions=predictions,
        references=references,
        examples=examples,
    )
    return metrics


def evaluate_forced_choice(model, raw_split, max_samples=None, batch_size=32):
    model.eval()
    eval_split = select_eval_split(raw_split, max_samples)
    device = next(model.parameters()).device
    label_token_ids = LABEL_TOKEN_IDS.to(device)
    predictions, references = [], []

    with temporary_padding_side(tokenizer, "left"), torch.inference_mode():
        for start in range(0, len(eval_split), batch_size):
            batch = eval_split[start : start + batch_size]
            inputs = tokenize_eval_batch(batch[TEXT_COLUMN], device)
            logits = model(**inputs).logits[:, -1, :].index_select(-1, label_token_ids)
            predictions.extend(logits.argmax(dim=-1).tolist())
            references.extend(int(label) for label in batch[LABEL_COLUMN])

    metrics = classification_metrics(predictions, references)
    metrics.update(predictions=predictions, references=references)
    return metrics


### 9. Baseline на полном validation split


До создания Prompt Tuning adapter измеряются две baseline-оценки на полном SST-2 validation split:

- generation-based evaluation;
- forced-choice evaluation.

Для обеих считаются accuracy, precision, recall, F1, Macro F1 и confusion matrix.


In [12]:
baseline_generation_metrics = (
    evaluate_generation(
        base_model,
        dataset["validation"],
        max_samples=(
            BASELINE_EVAL_SAMPLES
        ),
        batch_size=(
            GENERATION_BATCH_SIZE
        ),
    )
)

baseline_forced_choice_metrics = (
    evaluate_forced_choice(
        base_model,
        dataset["validation"],
        max_samples=(
            BASELINE_EVAL_SAMPLES
        ),
        batch_size=(
            FORCED_CHOICE_BATCH_SIZE
        ),
    )
)

assert (
    baseline_generation_metrics["total"]
    == len(dataset["validation"])
)

assert (
    baseline_forced_choice_metrics["total"]
    == len(dataset["validation"])
)

print_classification_summary(
    "Baseline — generation-based evaluation",
    baseline_generation_metrics,
)

print_classification_summary(
    "Baseline — forced-choice evaluation",
    baseline_forced_choice_metrics,
)

print(
    "\nGeneration examples:"
)

for (
    generated,
    prediction,
    reference,
) in (
    baseline_generation_metrics[
        "examples"
    ][:10]
):
    print(
        f"generated={generated!r:20s} "
        f"parsed={prediction!r:10s} "
        f"reference={reference}"
    )

RESULTS_TABLE.append(
    {
        "variant": "Исходная модель",
        "generation_accuracy": baseline_generation_metrics["accuracy"],
        "forced_choice_accuracy": baseline_forced_choice_metrics["accuracy"],
        "generation_macro_f1": baseline_generation_metrics["macro_f1"],
        "forced_choice_macro_f1": baseline_forced_choice_metrics["macro_f1"],
        "valid_output_rate": baseline_generation_metrics["valid_output_rate"],
    }
)



Baseline — generation-based evaluation
--------------------------------------
Accuracy:  3.10%
Macro F1:  0.0573
negative precision=0.0000 recall=0.0000 f1=0.0000
positive precision=1.0000 recall=0.0608 f1=0.1146
Valid output rate: 3.10%

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative             0         0       428
positive             0        27       417

Baseline — forced-choice evaluation
-----------------------------------
Accuracy:  51.49%
Macro F1:  0.3502
negative precision=1.0000 recall=0.0117 f1=0.0231
positive precision=0.5121 recall=1.0000 f1=0.6773

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative             5       423         0
positive             0       444         0

Generation examples:
generated='\n\n<think>\nHmm'   parsed=None       reference=positive
generated='\n\n<think>\nHmm'   parsed=None       reference=negative
generated='\n\n<think>\nHmm'   par

### 10. Prompt Tuning configuration


`PromptTuningConfig` создаёт набор обучаемых virtual tokens.

Используется text initialization: начальные embeddings virtual prompt получают смысловую инициализацию из `PROMPT_INIT_TEXT`.


In [13]:
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=(
        PromptTuningInit.TEXT
    ),
    num_virtual_tokens=(
        NUM_VIRTUAL_TOKENS
    ),
    prompt_tuning_init_text=(
        PROMPT_INIT_TEXT
    ),
    tokenizer_name_or_path=(
        MODEL_ID
    ),
)

print(peft_config)


PromptTuningConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.PROMPT_TUNING: 'PROMPT_TUNING'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, num_virtual_tokens=16, token_dim=None, num_transformer_submodules=None, num_attention_heads=None, num_layers=None, modules_to_save=None, prompt_tuning_init=<PromptTuningInit.TEXT: 'TEXT'>, prompt_tuning_init_text='Classify the sentiment of the movie review as positive or negative.', tokenizer_name_or_path='Qwen/Qwen3.5-2B-Base', tokenizer_kwargs=None)


### 11. Создание PeftModel


`get_peft_model()` связывает Prompt Tuning adapter с base model. После этого обучаемыми остаются virtual prompt embeddings.


In [14]:
model = get_peft_model(
    base_model,
    peft_config,
)

model.config.use_cache = False

total_params, trainable_params = (
    parameter_stats(model)
)

model.print_trainable_parameters()

print(
    f"\nTotal parameters:     "
    f"{total_params:,}"
)
print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)
print(
    f"Trainable share:      "
    f"{100 * trainable_params / total_params:.8f}%"
)


trainable params: 32,768 || all params: 1,881,857,856 || trainable%: 0.0017

Total parameters:     1,881,857,856
Trainable parameters: 32,768
Trainable share:      0.00174126%


### 12. Проверка frozen base weights


После `get_peft_model()` trainable parameters должны относиться только к Prompt Tuning adapter.


In [15]:
trainable_names = [
    name
    for name, parameter
    in model.named_parameters()
    if parameter.requires_grad
]

unexpected_trainable = [
    name
    for name in trainable_names
    if "prompt_encoder" not in name
]

print(
    f"Trainable tensors: "
    f"{len(trainable_names)}"
)

for name in trainable_names:
    print(
        " -",
        name,
    )

assert not unexpected_trainable, (
    "Unexpected trainable parameters:\n"
    + "\n".join(
        unexpected_trainable[:20]
    )
)

assert (
    trainable_params
    < total_params * 0.001
), (
    "Too many parameters are trainable "
    "for Prompt Tuning."
)


Trainable tensors: 1
 - prompt_encoder.default.embedding.weight


### 13. Linear LR scheduler и Early Stopping


Prompt Tuning использует существенно более высокий learning rate, чем LoRA, поэтому управление величиной optimizer step особенно важно.

Notebook явно использует Linear LR scheduler и Early Stopping по `eval_loss`.


#### Linear LR scheduler


Используется следующий schedule:

```text
warmup
→ learning rate растёт до LEARNING_RATE
→ после warmup линейно уменьшается к 0
```

При `WARMUP_STEPS = 0.05` первые 5% запланированных optimizer steps используются для warmup.

По мере обучения шаг становится меньше, что помогает точнее двигаться в области низкого loss и снижает риск продолжать делать одинаково крупные обновления на всём training run.


#### Early Stopping


Early Stopping контролирует `eval_loss`.

Используются:

```python
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.0
```

`patience=3` означает, что после трёх последовательных evaluation без улучшения `eval_loss` Trainer завершает training.

`threshold=0.0` позволяет учитывать любое реальное уменьшение validation loss.


#### Что если оптимальная область находится между validation points


Validation выполняется каждые `EVAL_STEPS = 100` optimizer steps, и checkpoint сохраняется с той же частотой.

Это заметно плотнее, чем evaluation только в конце каждой эпохи, и лучше подходит для Prompt Tuning с высоким learning rate.

Если фактический `eval_loss` будет резко колебаться между соседними точками, следующий контролируемый эксперимент — уменьшить `EVAL_STEPS` и `SAVE_STEPS` до 50, не меняя одновременно остальные hyperparameters.


### 14. TrainingArguments


Prompt Tuning сохраняет ранее проверенный learning rate `3e-2`, но scheduler теперь задаётся явно, а validation выполняется по optimizer steps.


In [16]:
use_bf16 = (
    device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

use_fp16 = (
    device.type == "cuda"
    and not use_bf16
)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    num_train_epochs=(
        NUM_TRAIN_EPOCHS
    ),
    per_device_train_batch_size=(
        TRAIN_BATCH_SIZE
    ),
    per_device_eval_batch_size=(
        EVAL_BATCH_SIZE
    ),
    gradient_accumulation_steps=(
        GRADIENT_ACCUMULATION_STEPS
    ),

    learning_rate=LEARNING_RATE,
    lr_scheduler_type=(
        LR_SCHEDULER_TYPE
    ),
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,

    logging_strategy="steps",
    logging_steps=20,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,

    bf16=use_bf16,
    fp16=use_fp16,
    tf32=(
        True
        if device.type == "cuda"
        else None
    ),

    optim=(
        "adamw_torch_fused"
        if device.type == "cuda"
        else "adamw_torch"
    ),

    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=(
        device.type == "cuda"
    ),

    include_for_metrics=["loss"],

    report_to="none",
    push_to_hub=False,
)

print(
    "Training examples:",
    len(dataset["train"]),
)
print(
    "Eval interval:",
    EVAL_STEPS,
    "optimizer steps",
)
print(
    "LR scheduler:",
    LR_SCHEDULER_TYPE,
)
print(
    "Early stopping patience:",
    EARLY_STOPPING_PATIENCE,
)
print(
    "Early stopping threshold:",
    EARLY_STOPPING_THRESHOLD,
)
print(
    "Checkpoint output:",
    CHECKPOINT_DIR,
)


Training examples: 4000
Eval interval: 100 optimizer steps
LR scheduler: linear
Early stopping patience: 3
Early stopping threshold: 0.0
Checkpoint output: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-prompt-tuning/checkpoints


#### Метрики промежуточной validation


Во время `Trainer` evaluation дополнительно считаются Label Token Accuracy и Perplexity.

Prompt Tuning добавляет virtual tokens перед исходной последовательностью. Поэтому перед token-level сравнением logits обрезаются до длины исходных labels.


In [17]:
def preprocess_logits_for_metrics(
    logits,
    labels,
):
    if isinstance(
        logits,
        tuple,
    ):
        logits = logits[0]

    if (
        logits.shape[1]
        != labels.shape[1]
    ):
        logits = logits[
            :,
            -labels.shape[1]:,
            :,
        ]

    return logits.argmax(
        dim=-1
    )


def compute_trainer_metrics(
    eval_prediction,
):
    predictions = (
        eval_prediction.predictions
    )
    labels = (
        eval_prediction.label_ids
    )

    if isinstance(
        predictions,
        tuple,
    ):
        predictions = (
            predictions[0]
        )

    predictions = np.asarray(
        predictions
    )
    labels = np.asarray(
        labels
    )

    shifted_predictions = (
        predictions[:, :-1]
    )
    shifted_labels = (
        labels[:, 1:]
    )

    valid_mask = (
        shifted_labels != -100
    )

    if (
        tokenizer.eos_token_id
        is not None
    ):
        valid_mask &= (
            shifted_labels
            != tokenizer.eos_token_id
        )

    valid_count = int(
        valid_mask.sum()
    )

    if valid_count:
        correct_count = int(
            (
                shifted_predictions[
                    valid_mask
                ]
                == shifted_labels[
                    valid_mask
                ]
            ).sum()
        )

        label_token_accuracy = (
            correct_count
            / valid_count
        )
    else:
        label_token_accuracy = 0.0

    metrics = {
        "label_token_accuracy": (
            label_token_accuracy
        )
    }

    losses = getattr(
        eval_prediction,
        "losses",
        None,
    )

    if losses is not None:
        losses = np.asarray(
            losses
        )

        mean_loss = float(
            losses.mean()
        )

        if np.isfinite(
            mean_loss
        ):
            metrics["perplexity"] = float(
                np.exp(
                    mean_loss
                )
            )

    return metrics


### 15. Trainer


Trainer получает `PeftModel`. Optimizer обновляет только parameters с `requires_grad=True`.


In [18]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=(
        processed_dataset["train"]
    ),
    eval_dataset=(
        processed_dataset["validation"]
    ),

    data_collator=data_collator,
    processing_class=tokenizer,

    compute_metrics=(
        compute_trainer_metrics
    ),
    preprocess_logits_for_metrics=(
        preprocess_logits_for_metrics
    ),

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=(
                EARLY_STOPPING_PATIENCE
            ),
            early_stopping_threshold=(
                EARLY_STOPPING_THRESHOLD
            ),
        )
    ],
)


### 16. Обучение


Для продолжения прерванного запуска можно использовать:

```python
trainer.train(resume_from_checkpoint=True)
```


#### Примечание о PAD/BOS/EOS tokens


При запуске Trainer может появиться информационное сообщение о синхронизации PAD/BOS/EOS tokens между tokenizer и model config.

В этом notebook `pad_token` при необходимости переиспользует существующий EOS token. Padding positions исключаются из supervised loss через `-100`, поэтому такое сообщение не означает ошибку training setup.


In [19]:
train_result = trainer.train()

trainer.log_metrics(
    "train",
    train_result.metrics,
)

trainer.save_metrics(
    "train",
    train_result.metrics,
)

trainer.save_state()

training_performance = dict(
    train_result.metrics
)

training_history = list(
    trainer.state.log_history
)

print(
    "\nTraining performance:"
)

for key in (
    "train_runtime",
    "train_samples_per_second",
    "train_steps_per_second",
    "train_loss",
):
    if key in training_performance:
        print(
            f"{key:28s}: "
            f"{training_performance[key]}"
        )


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 248044}.


Step,Training Loss,Validation Loss,Label Token Accuracy,Perplexity
100,0.113594,0.092559,0.931193,1.096978
200,0.101159,0.090258,0.936927,1.094457
300,0.104358,0.086163,0.930046,1.089984
400,0.097162,0.087920,0.935780,1.091901
500,0.068929,0.089276,0.938073,1.093382
600,0.066305,0.084720,0.943807,1.088412
700,0.058306,0.083342,0.942661,1.086914
750,0.056976,0.083539,0.943807,1.087128


***** train metrics *****
  epoch                    =        3.0
  total_flos               =  4969010GF
  train_loss               =     0.1076
  train_runtime            = 0:09:58.82
  train_samples_per_second =     20.039
  train_steps_per_second   =      1.252

Training performance:
train_runtime               : 598.8268
train_samples_per_second    : 20.039
train_steps_per_second      : 1.252
train_loss                  : 0.10764096395174662


### 17. Evaluation loss


После training выполняется evaluation лучшего checkpoint на полном validation split.

В progress-таблице и финальном evaluation доступны Validation Loss, Label Token Accuracy и Perplexity.


In [20]:
eval_metrics = (
    trainer.evaluate()
)

final_perplexity = (
    eval_metrics.get(
        "eval_perplexity"
    )
)

if final_perplexity is not None:
    final_perplexity = float(
        final_perplexity
    )

trainer.log_metrics(
    "eval",
    eval_metrics,
)

trainer.save_metrics(
    "eval",
    eval_metrics,
)

eval_metrics


Training Loss,Validation Loss,Step,Label Token Accuracy,Perplexity
0.056976,0.083342,750,0.942661,1.086914


***** eval metrics *****
  eval_label_token_accuracy = 0.9427
  eval_loss                 = 0.0833
  eval_perplexity           = 1.0869


{'eval_loss': 0.08334241807460785,
 'eval_label_token_accuracy': 0.9426605504587156,
 'eval_perplexity': 1.0869139238081615}

#### Training dynamics


`Trainer.state.log_history` содержит training loss, validation loss и learning rate.

Все три ряда отображаются на одном интерактивном Plotly-графике:

- train loss и validation loss используют левую Y-ось;
- learning rate использует правую Y-ось;
- общая X-ось показывает epoch.


In [21]:
train_loss_points = [
    (
        entry["epoch"],
        entry["loss"],
    )
    for entry in training_history
    if (
        "loss" in entry
        and "eval_loss" not in entry
        and "epoch" in entry
    )
]

eval_loss_points = [
    (
        entry["epoch"],
        entry["eval_loss"],
    )
    for entry in training_history
    if (
        "eval_loss" in entry
        and "epoch" in entry
    )
]

learning_rate_points = [
    (
        entry["epoch"],
        entry["learning_rate"],
    )
    for entry in training_history
    if (
        "learning_rate" in entry
        and "epoch" in entry
    )
]

training_fig = go.Figure()

if train_loss_points:
    (
        train_epochs,
        train_losses,
    ) = zip(
        *train_loss_points
    )

    training_fig.add_trace(
        go.Scatter(
            x=train_epochs,
            y=train_losses,
            mode="lines+markers",
            name="train loss",
            yaxis="y",
        )
    )

if eval_loss_points:
    (
        eval_epochs,
        eval_losses,
    ) = zip(
        *eval_loss_points
    )

    training_fig.add_trace(
        go.Scatter(
            x=eval_epochs,
            y=eval_losses,
            mode="lines+markers",
            name="validation loss",
            yaxis="y",
        )
    )

if learning_rate_points:
    (
        lr_epochs,
        learning_rates,
    ) = zip(
        *learning_rate_points
    )

    training_fig.add_trace(
        go.Scatter(
            x=lr_epochs,
            y=learning_rates,
            mode="lines",
            name="learning rate",
            yaxis="y2",
        )
    )

training_fig.update_layout(
    title="Training dynamics",
    height=430,

    xaxis={
        "title": "Epoch",
    },

    yaxis={
        "title": "Loss",
    },

    yaxis2={
        "title": "Learning rate",
        "overlaying": "y",
        "side": "right",
        "tickformat": ".1e",
        "showgrid": False,
    },

    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.02,
        "xanchor": "left",
        "x": 0,
    },

    margin={
        "l": 70,
        "r": 90,
        "t": 90,
        "b": 60,
    },
)

training_fig.show()


### 18. Accuracy после Prompt Tuning


После training повторяются оба baseline evaluation режима на том же полном validation split.

Это позволяет отдельно увидеть изменение task adaptation и изменение output behavior.


In [22]:
model = trainer.model
model.config.use_cache = True

final_generation_metrics = (
    evaluate_generation(
        model,
        dataset["validation"],
        max_samples=(
            FINAL_EVAL_SAMPLES
        ),
        batch_size=(
            GENERATION_BATCH_SIZE
        ),
    )
)

final_forced_choice_metrics = (
    evaluate_forced_choice(
        model,
        dataset["validation"],
        max_samples=(
            FINAL_EVAL_SAMPLES
        ),
        batch_size=(
            FORCED_CHOICE_BATCH_SIZE
        ),
    )
)

assert (
    final_generation_metrics["total"]
    == len(dataset["validation"])
)

assert (
    final_forced_choice_metrics["total"]
    == len(dataset["validation"])
)

print_classification_summary(
    "Prompt Tuning — generation-based evaluation",
    final_generation_metrics,
)

print_classification_summary(
    "Prompt Tuning — forced-choice evaluation",
    final_forced_choice_metrics,
)

print(
    "\nComparison:"
)

print(
    "Generation accuracy: "
    f"{baseline_generation_metrics['accuracy']:.2%}"
    " → "
    f"{final_generation_metrics['accuracy']:.2%}"
)

print(
    "Forced-choice accuracy: "
    f"{baseline_forced_choice_metrics['accuracy']:.2%}"
    " → "
    f"{final_forced_choice_metrics['accuracy']:.2%}"
)

print(
    "Generation Macro F1: "
    f"{baseline_generation_metrics['macro_f1']:.4f}"
    " → "
    f"{final_generation_metrics['macro_f1']:.4f}"
)

print(
    "Forced-choice Macro F1: "
    f"{baseline_forced_choice_metrics['macro_f1']:.4f}"
    " → "
    f"{final_forced_choice_metrics['macro_f1']:.4f}"
)

print(
    "Valid output rate: "
    f"{baseline_generation_metrics['valid_output_rate']:.2%}"
    " → "
    f"{final_generation_metrics['valid_output_rate']:.2%}"
)

print(
    "\nGeneration examples:"
)

for (
    generated,
    prediction,
    reference,
) in (
    final_generation_metrics[
        "examples"
    ][:10]
):
    print(
        f"generated={generated!r:20s} "
        f"parsed={prediction!r:10s} "
        f"reference={reference}"
    )

artifact_expected_generation = (
    final_generation_metrics[
        "predictions"
    ][:ARTIFACT_RELOAD_SAMPLES]
)

artifact_expected_forced_choice = (
    final_forced_choice_metrics[
        "predictions"
    ][:ARTIFACT_RELOAD_SAMPLES]
)

RESULTS_TABLE.append(
    {
        "variant": "Prompt Tuning",
        "generation_accuracy": final_generation_metrics["accuracy"],
        "forced_choice_accuracy": final_forced_choice_metrics["accuracy"],
        "generation_macro_f1": final_generation_metrics["macro_f1"],
        "forced_choice_macro_f1": final_forced_choice_metrics["macro_f1"],
        "valid_output_rate": final_generation_metrics["valid_output_rate"],
    }
)


/usr/local/lib/python3.10/dist-packages/peft/peft_model.py:1657: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")



Prompt Tuning — generation-based evaluation
-------------------------------------------
Accuracy:  94.27%
Macro F1:  0.9427
negative precision=0.9355 recall=0.9486 f1=0.9420
positive precision=0.9498 recall=0.9369 f1=0.9433
Valid output rate: 100.00%

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative           406        22         0
positive            28       416         0

Prompt Tuning — forced-choice evaluation
----------------------------------------
Accuracy:  94.27%
Macro F1:  0.9427
negative precision=0.9335 recall=0.9509 f1=0.9421
positive precision=0.9518 recall=0.9347 f1=0.9432

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative           407        21         0
positive            29       415         0

Comparison:
Generation accuracy: 3.10% → 94.27%
Forced-choice accuracy: 51.49% → 94.27%
Generation Macro F1: 0.0573 → 0.9427
Forced-choice Macro F1: 0.3502 → 0.9427
Val

In [23]:
del trainer
del train_result
del eval_metrics

clear_device_memory()


### 19. Сохранение PEFT adapter


В `OUTPUT_DIR` сохраняются Prompt Tuning adapter и tokenizer.

Base model не копируется: она продолжает использоваться отдельно.


In [24]:
model.save_pretrained(
    OUTPUT_DIR
)

tokenizer.save_pretrained(
    OUTPUT_DIR
)

print(
    f"Saved Prompt Tuning adapter to: "
    f"{OUTPUT_DIR}"
)


Saved Prompt Tuning adapter to: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-prompt-tuning


#### Reproducibility metadata


Помимо seed notebook фиксирует revisions модели и dataset repository, fingerprints реально использованных splits и training configuration.


In [25]:
import json as json_module

REPRODUCIBILITY_PATH = (
    OUTPUT_DIR
    / "reproducibility.json"
)

reproducibility_data = {
    "seed": SEED,
    "run_mode": RUN_MODE,

    "model_id": MODEL_ID,
    "model_revision_requested": (
        MODEL_REVISION
    ),
    "model_revision_resolved": (
        resolved_model_revision
    ),

    "dataset_id": DATASET_ID,
    "dataset_revision_requested": (
        DATASET_REVISION
    ),
    "dataset_revision_resolved": (
        resolved_dataset_revision
    ),

    "train_fingerprint": (
        dataset_train_fingerprint
    ),
    "validation_fingerprint": (
        dataset_validation_fingerprint
    ),

    "train_examples": (
        len(dataset["train"])
    ),
    "validation_examples": (
        len(dataset["validation"])
    ),

    "method": "prompt_tuning",
    "num_virtual_tokens": (
        NUM_VIRTUAL_TOKENS
    ),
    "prompt_init_text": (
        PROMPT_INIT_TEXT
    ),

    "learning_rate": (
        LEARNING_RATE
    ),
    "lr_scheduler_type": (
        LR_SCHEDULER_TYPE
    ),
    "warmup_steps": (
        WARMUP_STEPS
    ),
    "weight_decay": (
        WEIGHT_DECAY
    ),

    "eval_steps": EVAL_STEPS,
    "save_steps": SAVE_STEPS,

    "early_stopping_patience": (
        EARLY_STOPPING_PATIENCE
    ),
    "early_stopping_threshold": (
        EARLY_STOPPING_THRESHOLD
    ),
}

REPRODUCIBILITY_PATH.write_text(
    json_module.dumps(
        reproducibility_data,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    "Saved reproducibility metadata: "
    f"{REPRODUCIBILITY_PATH}"
)


Saved reproducibility metadata: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-prompt-tuning/reproducibility.json


### 20. Размер Prompt Tuning adapter


Размер считается только по adapter files, без Trainer checkpoints.


In [26]:
adapter_files = [
    OUTPUT_DIR
    / "adapter_model.safetensors",

    OUTPUT_DIR
    / "adapter_config.json",
]

adapter_size_bytes = sum(
    path.stat().st_size
    for path in adapter_files
    if path.exists()
)

print(
    f"Prompt Tuning adapter size: "
    f"{adapter_size_bytes / 1024**2:.3f} MiB"
)

print(
    "\nFinal export files:"
)

for file in sorted(
    OUTPUT_DIR.iterdir()
):
    if file.is_file():
        print(
            f"{file.name:32s} "
            f"{file.stat().st_size / 1024:.1f} KiB"
        )


Prompt Tuning adapter size: 0.126 MiB

Final export files:
README.md                        5.1 KiB
adapter_config.json              0.6 KiB
adapter_model.safetensors        128.1 KiB
chat_template.jinja              7.6 KiB
reproducibility.json             0.8 KiB
tokenizer.json                   19521.1 KiB
tokenizer_config.json            1.1 KiB


In [27]:
del model
del base_model

clear_device_memory()


### 21. Проверка сохранённого adapter


Сохранённый Prompt Tuning adapter загружается заново поверх чистой base model.

Проверка выполняется по умолчанию на нескольких фиксированных validation examples и сравнивает generation и forced-choice predictions до сохранения и после reload.


In [28]:
if RUN_ARTIFACT_RELOAD_TEST:
    adapter_config = (
        PeftConfig.from_pretrained(
            OUTPUT_DIR
        )
    )

    reloaded_base_model = (
        AutoModelForCausalLM
        .from_pretrained(
            adapter_config
            .base_model_name_or_path,
            revision=MODEL_REVISION,
            dtype=model_dtype,
        )
    )

    reloaded_model = (
        PeftModel.from_pretrained(
            reloaded_base_model,
            OUTPUT_DIR,
        )
    )

    reloaded_model = (
        reloaded_model.to(device)
    )

    reloaded_model.eval()

    reload_generation_metrics = (
        evaluate_generation(
            reloaded_model,
            dataset["validation"],
            max_samples=(
                ARTIFACT_RELOAD_SAMPLES
            ),
            batch_size=(
                ARTIFACT_RELOAD_SAMPLES
            ),
        )
    )

    reload_forced_choice_metrics = (
        evaluate_forced_choice(
            reloaded_model,
            dataset["validation"],
            max_samples=(
                ARTIFACT_RELOAD_SAMPLES
            ),
            batch_size=(
                ARTIFACT_RELOAD_SAMPLES
            ),
        )
    )

    assert (
        reload_generation_metrics[
            "predictions"
        ]
        == artifact_expected_generation
    ), (
        "Generation predictions changed "
        "after adapter reload."
    )

    assert (
        reload_forced_choice_metrics[
            "predictions"
        ]
        == artifact_expected_forced_choice
    ), (
        "Forced-choice predictions changed "
        "after adapter reload."
    )

    print(
        "Adapter reload smoke test: "
        "PASSED"
    )
    print(
        f"Checked examples: "
        f"{ARTIFACT_RELOAD_SAMPLES}"
    )

    del reload_generation_metrics
    del reload_forced_choice_metrics
    del reloaded_model
    del reloaded_base_model
    del adapter_config

    clear_device_memory()
else:
    print(
        "RUN_ARTIFACT_RELOAD_TEST=False — "
        "reload test skipped."
    )


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Adapter reload smoke test: PASSED
Checked examples: 8


### 22. Создание Hugging Face Model Card


Model Card создаётся как компактный `README.md` для Prompt Tuning adapter.

В карточке остаются базовые сведения о методе, основные результаты, пример загрузки и короткие ограничения.


In [29]:
MODEL_CARD_PATH = (
    OUTPUT_DIR
    / "README.md"
)

model_display_name = (
    HUB_MODEL_ID.split("/")[-1]
)

evaluation_is_full_validation = (
    final_generation_metrics["total"]
    == len(raw_dataset["validation"])
)

evaluation_scope = (
    "full SST-2 validation split"
    if evaluation_is_full_validation
    else (
        f"{final_generation_metrics['total']} "
        "SST-2 validation examples"
    )
)

perplexity_text = (
    f"{final_perplexity:.4f}"
    if final_perplexity is not None
    else "n/a"
)

card_text = f"""---
base_model: {MODEL_ID}
library_name: peft
pipeline_tag: text-generation
datasets:
- {DATASET_ID}
language:
- en
license: apache-2.0
tags:
- peft
- prompt-tuning
- qwen3.5
- sentiment-analysis
---

# {model_display_name}

Prompt Tuning adapter for
[`{MODEL_ID}`](https://huggingface.co/{MODEL_ID}),
fine-tuned on
[`{DATASET_ID}`](https://huggingface.co/datasets/{DATASET_ID})
for binary sentiment classification.

The adapter predicts `positive` or `negative`.

## Model Details

| Property | Value |
|---|---|
| Base model | `{MODEL_ID}` |
| Method | Prompt Tuning |
| Dataset | `{DATASET_ID}` |
| Task | Sentiment classification |
| Labels | `negative`, `positive` |
| Virtual tokens | {NUM_VIRTUAL_TOKENS} |
| Prompt initialization | `{PROMPT_INIT_TEXT}` |
| Adapter size | {adapter_size_bytes / 1024**2:.2f} MiB |

## Evaluation

Evaluation scope: **{evaluation_scope}**.

| Metric | Base model | Prompt Tuning |
|---|---:|---:|
| Generation accuracy | {baseline_generation_metrics["accuracy"]:.2%} | **{final_generation_metrics["accuracy"]:.2%}** |
| Forced-choice accuracy | {baseline_forced_choice_metrics["accuracy"]:.2%} | **{final_forced_choice_metrics["accuracy"]:.2%}** |
| Generation Macro F1 | {baseline_generation_metrics["macro_f1"]:.4f} | **{final_generation_metrics["macro_f1"]:.4f}** |
| Forced-choice Macro F1 | {baseline_forced_choice_metrics["macro_f1"]:.4f} | **{final_forced_choice_metrics["macro_f1"]:.4f}** |
| Perplexity | — | {perplexity_text} |

## Usage

```python
import torch
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

BASE_MODEL_ID = "{MODEL_ID}"
ADAPTER_ID = "{HUB_MODEL_ID}"

tokenizer = (
    AutoTokenizer.from_pretrained(
        BASE_MODEL_ID
    )
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = (
        tokenizer.eos_token
    )

base_model = (
    AutoModelForCausalLM
    .from_pretrained(
        BASE_MODEL_ID,
        dtype="auto",
    )
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_ID,
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

review = (
    "a wonderfully acted "
    "and moving story"
)

prompt = (
    "{VISIBLE_INSTRUCTION}\\n"
    f"Review: {{review}}\\n"
    "Sentiment:"
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens={MAX_NEW_TOKENS},
        do_sample=False,
        pad_token_id=(
            tokenizer.eos_token_id
        ),
    )

generated = outputs[
    :,
    inputs["input_ids"].shape[1]:,
]

prediction = tokenizer.decode(
    generated[0],
    skip_special_tokens=True,
).strip()

print(prediction)
```

## Limitations

- Designed for English SST-2 sentiment classification.
- Requires the documented prompt format and the base model `{MODEL_ID}`.
"""

MODEL_CARD_PATH.write_text(
    card_text,
    encoding="utf-8",
)

print(
    f"Saved Model Card: "
    f"{MODEL_CARD_PATH}"
)


Saved Model Card: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-prompt-tuning/README.md


### 23. Публикация в Hugging Face Hub


Push отключён по умолчанию.

Перед публикацией выполните `hf auth login` и установите `PUSH_TO_HUB=True`.

На Hub отправляются Prompt Tuning adapter, tokenizer, `README.md` и `reproducibility.json`. Trainer checkpoints исключаются.


In [30]:
if PUSH_TO_HUB:
    create_repo(
        repo_id=HUB_MODEL_ID,
        repo_type="model",
        exist_ok=True,
    )

    api = HfApi()

    commit_info = (
        api.upload_folder(
            folder_path=OUTPUT_DIR,
            repo_id=HUB_MODEL_ID,
            repo_type="model",
            ignore_patterns=[
                "checkpoints/**",
                "checkpoint-*/**",
                "runs/**",
                "*.pt",
                "*.pth",
            ],
            commit_message=(
                "Upload Qwen3.5-2B "
                "SST-2 Prompt Tuning adapter "
                "and model card"
            ),
        )
    )

    print(
        f"Published: "
        f"https://huggingface.co/"
        f"{HUB_MODEL_ID}"
    )

    print(
        f"Commit: "
        f"{commit_info.commit_url}"
    )
else:
    print(
        "PUSH_TO_HUB=False - "
        "nothing was uploaded."
    )


PUSH_TO_HUB=False - nothing was uploaded.


## Результаты

### Итоговая сравнительная таблица

In [31]:
assert [row["variant"] for row in RESULTS_TABLE] == [
    "Исходная модель",
    "Prompt Tuning",
]

print(
    f"{'Вариант':18s}"
    f"{'Generation Accuracy':>22s}"
    f"{'Forced-Choice Accuracy':>24s}"
    f"{'Generation Macro F1':>22s}"
    f"{'Forced-Choice Macro F1':>24s}"
    f"{'Valid Output Rate':>20s}"
)

print("-" * 130)

for row in RESULTS_TABLE:
    print(
        f"{row['variant']:18s}"
        f"{row['generation_accuracy']:22.2%}"
        f"{row['forced_choice_accuracy']:24.2%}"
        f"{row['generation_macro_f1']:22.4f}"
        f"{row['forced_choice_macro_f1']:24.4f}"
        f"{row['valid_output_rate']:20.2%}"
    )


Вариант              Generation Accuracy  Forced-Choice Accuracy   Generation Macro F1  Forced-Choice Macro F1   Valid Output Rate
----------------------------------------------------------------------------------------------------------------------------------
Исходная модель                    3.10%                  51.49%                0.0573                  0.3502               3.10%
Prompt Tuning                     94.27%                  94.27%                0.9427                  0.9427             100.00%


Сравнительная таблица показывает существенное улучшение качества после Prompt Tuning сразу в обоих режимах оценки.

У исходной модели **Generation Accuracy составляет 3.10%**, а **Valid Output Rate — 3.10%**. При свободной генерации базовая модель почти всегда нарушает требуемый формат ответа: вместо `positive` или `negative` она продолжает промпт произвольным текстом. Поэтому низкую Generation Accuracy нельзя интерпретировать как прямую оценку способности исходной модели распознавать тональность.

**Forced-Choice Accuracy исходной модели составляет 51.49%**, а **Forced-Choice Macro F1 — 0.3502**. Accuracy находится примерно на уровне случайного выбора для бинарной классификации, а существенно более низкий Macro F1 указывает на выраженный перекос в сторону одного из классов. Следовательно, проблема исходной модели связана не только с форматом генерации: без адаптации она также слабо разделяет классы SST-2 в выбранной постановке задачи.

После Prompt Tuning **Generation Accuracy возрастает до 94.27%**, а **Forced-Choice Accuracy — также до 94.27%**. Прирост относительно исходной модели составляет соответственно **+91.17** и **+42.78 процентного пункта**. При этом **Generation Macro F1 увеличивается с 0.0573 до 0.9427**, а **Forced-Choice Macro F1 — с 0.3502 до 0.9427**.

**Valid Output Rate возрастает с 3.10% до 100.00%**, то есть после Prompt Tuning модель стабильно соблюдает требуемый формат ответа. Особенно показательно, что после адаптации **Generation Accuracy и Forced-Choice Accuracy полностью совпадают — 94.27%**, как и соответствующие значения **Macro F1 — 0.9427**. Это означает, что свободная генерация больше практически не вносит дополнительной ошибки относительно принудительного выбора класса.

Таким образом, в этом эксперименте Prompt Tuning успешно решил две задачи одновременно: существенно улучшил качество бинарной классификации SST-2 и сформировал устойчивое поведение при генерации ответа. При этом базовые параметры модели остаются замороженными, а адаптация выполняется только за счёт небольшого набора обучаемых виртуальных токенов.

## Источники


- Qwen3.5-2B-Base: https://huggingface.co/Qwen/Qwen3.5-2B-Base
- Transformers Qwen3.5: https://huggingface.co/docs/transformers/model_doc/qwen3_5
- Transformers Trainer: https://huggingface.co/docs/transformers/main_classes/trainer
- Transformers optimizer schedules: https://huggingface.co/docs/transformers/main_classes/optimizer_schedules
- Transformers callbacks: https://huggingface.co/docs/transformers/main_classes/callback
- PEFT Prompt Tuning: https://huggingface.co/docs/peft/main/package_reference/prompt_tuning
- PEFT causal LM Prompt Tuning guide: https://huggingface.co/docs/peft/main/task_guides/clm-prompt-tuning
- Stanford SST-2: https://huggingface.co/datasets/stanfordnlp/sst2
